Step 1: Setup

In [ ]:
import os
from openai import AzureOpenAI
import json

In [ ]:
def call_LLM(system_content = "you are a helpful assistant", user_content = ""):
    os.environ['AZURE_OPENAI_API_KEY'] = "XXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX"
    os.environ['AZURE_OPENAI_ENDPOINT'] = 'http://pluralsight.openai.azure.com'
    os.environ['AZURE_OPENAI_API_VERSION'] = '2024-06-01'
    client = AzureOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),  
    )
    response = client.chat.completions.create( 
        model="gpt-4o-mini", 
        messages=[ 
            { 
                "role": "system", 
                "content": f"{system_content}"
            },
            { 
                "role": "user", 
                "content": f"{user_content}"
            }
        ]
    ) 
    return response

In [ ]:
example = call_LLM(user_content = "In less than 10 words, why do you like AI?")

In [ ]:
print(example.choices[0].message.content)

Step 2: Clarity, Specificity, and Iteration

In [ ]:
# 2.1 Create a prompt descripting seatles tourist attrictions.
example = call_LLM(
    ## user_content = "Tell me about that city, e.g. culture, food, language, education located with tower eiffel"
    user_content = "Tell me about that city, e.g. culture, food, language, education located with tower eiffel. Please provide a JSON structure with a top level attractions containing a lust of objects, each with name and description."
)

In [ ]:
## print(example.choices[0].message.content)
print(json.dumps(example.choices[0].message.content, indent=4))

Step 3: Structured Prompting with RTCF

In [ ]:
editingGuidelines = """1. Clarity and Conciseness

1.1 Eliminate unnecessary words and redundancy. Prefer shorter sentences when clarity improves.
1.2 Avoid jargon unless the target audience is specialized and expects domain-specific terminology.
1.3 Replace vague words (“stuff,” “things,” “utilize,” “various,” “numerous”) with specific and direct language.
1.4 If a sentence contains more than one idea, split it into two independent sentences where appropriate.

2. Structure and Flow

2.1 Organize content using a logical progression: context → explanation → example → conclusion.
2.2 Every paragraph must have a single, identifiable purpose. If a paragraph shifts topics midway, break it into two.
2.3 Use transitions between sentences and paragraphs to maintain narrative coherence (e.g., “however,” “for example,” “in contrast”).
2.4 Place the most important information near the beginning of each section (“front-loading”).

3. Tone and Voice

3.1 Maintain a professional, friendly, and instructive tone.
3.2 Avoid overly casual expressions unless the publication’s voice explicitly encourages informality.
3.3 Use active voice whenever possible. Passive voice is acceptable only when the actor is unknown, irrelevant, or distracting.
3.4 Avoid intensifiers such as “very,” “really,” and “extremely” when they weaken the message instead of strengthening it.

4. Grammar and Mechanics

4.1 Follow American English spelling conventions unless otherwise specified.
4.2 Serial commas (Oxford commas) are required in all lists of three or more items.
4.3 Numbers 0-9 should be written as words; numbers 10 and above should be numeric, unless part of a data table or technical spec.
4.4 Avoid ambiguous pronoun references. A pronoun must clearly refer to a single, identifiable noun.

5. Formatting Standards

5.1 Headings should use title case; subheadings should use sentence case.
5.2 Bullet lists must be parallel in structure (e.g., all verbs start in the same form).
5.3 Bold is reserved for key terms or definitions; italics should be used only for emphasis or publication titles.
5.4 Avoid over-formatting; visual hierarchy should guide the reader without distraction.

6. Examples of Acceptable Revisions

Original:
“The system utilizes numerous features that help with efficiency in various situations.”
Revised:
“The system includes several features that improve efficiency in specific scenarios.”

7. Common Issues to Flag

Run-on sentences

Inconsistent tense usage

Misaligned tone for the audience

Overuse of buzzwords

Lack of concrete examples

Editors should document major changes in revision notes and flag areas where requirements conflict with these guidelines."""

In [ ]:
# Task 3.1: Apply the RTCF structure
role = "You are a helpful data analyst"
task = "Please return the specific rule about how to handle numbers"
context = "Here are the specific rules: " + editingGuidelines
returnFormat = " with the output format should follow the specific rule mentioned"

example = call_LLM(
    system_content = f"{role}",
    user_content = f"{context}, {task} {returnFormat}"
)

In [ ]:
print(example.choices[0].message.content)

Step 4: Multi-Turn Conversations

In [ ]:
from openai import AzureOpenAI
import os

def call_LLM_loop(system_content="You are a helpful assistant."):
    os.environ['AZURE_OPENAI_API_KEY'] = "XXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX"
    os.environ['AZURE_OPENAI_ENDPOINT'] = 'http://pluralsight.openai.azure.com'
    os.environ['AZURE_OPENAI_API_VERSION'] = '2024-06-01'
    client = AzureOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),  
    )
    
    # Initialize conversation history
    messages =  [ 
            { 
                "role": "system", 
                "content": f"{system_content}"
            }
        ]
    
    print("Type your message (or 'exit' to quit):")
    while True:
        user_content = input("You: ")
        if user_content.lower() in ["exit", "quit"]:
            print("Conversation ended.")
            break
        
        messages.append({"role": "user", "content": user_content})
        
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )
        
        reply = response.choices[0].message.content
        print(f"AI: {reply}\n")
        
        # Save the AI response in context
        messages.append({"role": "assistant", "content": reply})

    return messages

In [ ]:
call_LLM_loop()

Step 5: Responsible and Safe Prompting

In [ ]:
# Task 5.2: Mitigate hallucinations
system_context = "You are a techer. If you don't know the answer, say 'I don't know'"
question = "When did the first dinosaur become president of the moon?"
example = call_LLM(system_content = system_context, user_content = question)